# SITCOM-2135 - Dew Point Investigation

## Input Parameters

In [ ]:
day_obs_start = 20240501
day_obs_end = 20240801
sampling = "1h"

## Setup Notebook

In [ ]:
import asyncio
import logging
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.lines import Line2D

from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)
from astropy.time import Time

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Constants used in the notebook
ess_weather_station_sal_index = 301
m1m3_inside_cell_sal_index = 113
dome_inside_sal_index = 111

# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

## Helper Functions

In [ ]:
async def query_ess_weather_station(client, start_time, end_time, sampling="1h"):
    """
    Query the actual temperature from the weather station.
    """
    if not isinstance(start_time, Time):
        start_time = Time(start_time.to_pydatetime(), scale="utc")
    if not isinstance(end_time, Time):
        end_time = Time(end_time.to_pydatetime(), scale="utc")

    query = f"""
        SELECT
            max(dewPointItem) AS maxDewPoint,
            mean(dewPointItem) AS meanDewPoint,
            min(dewPointItem) AS minDewPoint
        FROM "lsst.sal.ESS.dewPoint"
        WHERE time > '{start_time.isot}Z'
        AND time < '{end_time.isot}Z'
        AND salIndex = {ess_weather_station_sal_index}
        GROUP BY time({sampling})
        """

    _df = await client.influx_client.query(query)
    
    return _df

## Analysis

In [ ]:
t_start = getDayObsStartTime(day_obs_start)
t_end = getDayObsEndTime(day_obs_end)

df = await query_ess_weather_station(efd_client, t_start, t_end, sampling)

In [ ]:
df

In [ ]:
title = f"Time Series - DewPoint\nFrom {day_obs_start} to {day_obs_end} - Weather Station ESS.{ess_weather_station_sal_index}"
fig, ax = plt.subplots(num=title)

ax.fill_between(df.index, df["minDewPoint"], df["maxDewPoint"], fc="C1", alpha=0.75, label="Min/Max Dew Points")
ax.plot(df["meanDewPoint"], label=f"Mean Dew Point", color="C0")
ax.legend()

t_start = getDayObsStartTime(day_obs_start).to_datetime()
t_end = getDayObsEndTime(day_obs_end).to_datetime()

ax.set_xlim(t_start, t_end)
ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature [deg C]")

fig.suptitle(title)
fig.tight_layout()
fig.autofmt_xdate()

plt.savefig(f"{title.replace('/n', ' ')}.png")
plt.show()

In [ ]:
title = f"Histogram - DewPoint\nFrom {day_obs_start} to {day_obs_end} - Weather Station ESS.{ess_weather_station_sal_index}"
fig, ax = plt.subplots(num=title)

ax.hist(df["meanDewPoint"], label=f"Mean Dew Point", color="C0", bins=50)
ax.legend()

fig.suptitle(title)
fig.tight_layout()
fig.autofmt_xdate()

plt.savefig(f"{title.replace('/n', ' ')}.png")
plt.show()